# AI Driven DevOps with Docker

**Learning Objectives**

By the end of this lab, you will be able to:

1. Use GitHub Copilot and Azure OpenAI to design DevOps pipelines
2. Generate production-ready Dockerfiles using AI prompts
3. Debug container issues with AI guidance
4. Deploy microservices using AI-generated workflows
5. Automate CI/CD tasks using reusable prompt templates
6. Use Terraform configurations generated and reviewed with AI assistance
7. Walk through a full development-to-deployment cycle

**How this notebook is structured**

Each section follows the same pattern: a markdown cell explains the concept and what to expect, followed by one or more code cells you run in order. Every code cell is self-contained and produces visible output so you can verify progress before moving on.

**AI-DevOps Workflow**

```
Developer Intent
      |
      v
AI Prompt  (GitHub Copilot inside VS Code  OR  Azure OpenAI via this notebook)
      |
      v
Generated Artifact  (Dockerfile / Jenkinsfile / Terraform / docker-compose.yml)
      |
      v
Human Review + Refinement  <-- this step is never optional
      |
      v
Build / Test / Deploy
```

## Section 1. Introduction and Setup

This section verifies that all required tools are reachable from the notebook kernel and installs the Python packages used throughout the lab.

**Tooling used in this lab**

- **Azure OpenAI** - programmatic AI calls from notebook cells via the `openai` Python SDK
- **Docker** - builds container images and runs containers on this machine
- **Jenkins** - CI/CD automation server; we generate a Jenkinsfile for it in Section 6
- **VS Code** - primary editor; also runs this notebook via the Jupyter extension

Run the four cells below in order. Each one must complete without errors before you continue.

In [ ]:
# Verify Python, Docker, and Git are all accessible from this kernel.
# If Docker shows an error, make sure Docker Desktop is running (check the system tray).
import sys, subprocess

print(f"Python: {sys.version}")

docker_ver = subprocess.run(
    ["docker", "--version"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(f"Docker: {docker_ver.stdout.strip() or docker_ver.stderr.strip()}")

git_ver = subprocess.run(
    ["git", "--version"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(f"Git:    {git_ver.stdout.strip()}")

In [ ]:
# Install the three packages used throughout this lab.
# openai      - Azure OpenAI SDK
# python-dotenv - loads credentials from the .env file
# requests    - used for the smoke test HTTP call in Section 3
%pip install openai python-dotenv requests --quiet

In [ ]:
# Load Azure OpenAI credentials from the .env file in the same directory as this notebook.
# The .env file must contain:
#   AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/
#   AZURE_OPENAI_API_KEY=<key>
#   AZURE_OPENAI_API_VERSION=2024-02-01
#   DEPLOYMENT_NAME=<deployment>
#
# OUTPUT_DIR is the folder where all generated files (Dockerfile, Jenkinsfile, etc.) are written.
# Using a subfolder keeps the workspace clean and makes it easy to inspect or delete all outputs.
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
DEPLOYMENT = os.environ["DEPLOYMENT_NAME"]

OUTPUT_DIR = Path("lab_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Azure OpenAI client ready.")
print("Output directory:", OUTPUT_DIR.resolve())

In [ ]:
# Two helper functions used throughout every section of this lab.
#
# ask_ai(prompt)  - sends a prompt to Azure OpenAI and returns the raw text response.
#                   The system prompt primes the model as a DevOps expert to keep
#                   responses technical and concise.
#
# strip_fences(text) - removes markdown code fences (```dockerfile, ```yaml, etc.)
#                   that the model sometimes wraps around file content. If these
#                   are written to disk, tools like Docker and Terraform fail to
#                   parse the file. Always call strip_fences() before writing
#                   AI output to disk.
import re

def ask_ai(prompt: str, system: str = "You are a DevOps expert. Be concise and precise.") -> str:
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

def strip_fences(text: str) -> str:
    return re.sub(r'^```[\w]*\n?', '', text.strip(), flags=re.MULTILINE).rstrip('`').strip()

print("Helpers ready: ask_ai, strip_fences")

## Section 2. Designing DevOps Pipelines with Generative AI

Before writing any configuration file, it helps to ask the AI to sketch the overall pipeline. This gives you a plan to validate against your real infrastructure before committing to code.

Generative AI contributes at three stages of pipeline design:

1. **Planning** - describe the app and constraints; get a numbered stage breakdown in seconds
2. **Scaffolding** - turn each planned stage into a concrete config file (Dockerfile, Jenkinsfile, etc.)
3. **Review** - paste an existing config and ask the AI to critique it for security or correctness

The sample application is a minimal Python Flask service with a single health-check endpoint. We define it inline so this notebook is fully self-contained and does not depend on external files.

In [ ]:
# Write the Flask application and its dependencies to lab_output/.
# app.py      - a single-route Flask app that returns a JSON health response
# requirements.txt - pinned versions of Flask and Gunicorn
#
# Pinning exact versions (flask==3.0.3 rather than flask>=3) prevents the image
# from silently picking up a newer version that might break something.
app_py = '''\
from flask import Flask, jsonify

app = Flask(__name__)

@app.route("/")
def health():
    return jsonify(status="ok", service="ai-devops-demo")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

requirements_txt = "flask==3.0.3\ngunicorn==22.0.0\n"

(OUTPUT_DIR / "app.py").write_text(app_py)
(OUTPUT_DIR / "requirements.txt").write_text(requirements_txt)

print("Written to", OUTPUT_DIR)
print("  app.py")
print("  requirements.txt")

In [ ]:
# Ask the AI to produce a high-level CI/CD pipeline plan for this app.
# The prompt is deliberately concise: we give just enough context (language, deployment
# target) for the model to produce a relevant plan without over-specifying.
# The output is a numbered list of stages we will implement in Sections 3-7.
pipeline_prompt = """\
I have a Python Flask microservice.
Generate a high-level CI/CD pipeline plan covering: build, unit test, container image scan,
Docker image build and push to a registry, and SSH deployment to a Linux VM.
Format as numbered stages. Be brief.
"""

pipeline_plan = ask_ai(pipeline_prompt)
print(pipeline_plan)

**Review the plan above before moving on.**

The AI output is a starting point, not a finished spec. Work through these questions:

1. Which stages map directly to your real infrastructure? (Does your registry match? Does your deploy host use SSH?)
2. Which stages need credentials or environment-specific values that the AI cannot know?
3. What is missing for your security or compliance requirements? (Secrets scanning? SAST? Approval gates?)

In the next section we move from plan to implementation, starting with the Dockerfile.

## Section 3. Generating a Dockerfile with AI

A Dockerfile is the specification for your container image. A well-structured Dockerfile uses multiple build stages to keep the final image small and secure:

- **Stage 1 (builder):** install all dependencies, including build tools that are not needed at runtime
- **Stage 2 (runtime):** copy only the installed packages and application source; run as a non-root user

The key to getting good output from the AI is a structured, constrained prompt. We specify language, framework, port, entrypoint, and explicit requirements. The final line, "Output only the Dockerfile, no explanation," prevents the model from wrapping the file in prose or markdown.

After generating the file we build it, run it, and verify the endpoint responds correctly.

In [ ]:
# Generate the Dockerfile using a structured prompt.
# strip_fences() removes any ```dockerfile ... ``` wrapper the model may add.
# Without strip_fences, Docker fails to parse the file on line 1.
dockerfile_prompt = """\
Generate a multi-stage Dockerfile for the following application:
- Language: Python 3.12
- Framework: Flask with Gunicorn
- Port: 5000
- Entrypoint: gunicorn -w 2 -b 0.0.0.0:5000 app:app
- Dependencies file: requirements.txt

Requirements:
1. Stage 1 (builder): install dependencies into a venv
2. Stage 2 (runtime): copy only the venv and app source; run as a non-root user
3. No unnecessary packages in the final image
4. Output only the Dockerfile, no explanation.
"""

dockerfile_content = strip_fences(ask_ai(dockerfile_prompt))
(OUTPUT_DIR / "Dockerfile").write_text(dockerfile_content)

print("Generated Dockerfile:")
print("-" * 60)
print(dockerfile_content)

**Review the Dockerfile before building.** Check that:

1. The base image tag is a real, published tag (e.g., `python:3.12-slim` not `python:3.99-slim`)
2. The runtime stage copies from the builder stage, not from the internet
3. There is a `USER` instruction switching to a non-root user before `CMD`
4. No secrets, credentials, or `.env` files are copied into the image

**Using GitHub Copilot for the same task (VS Code)**

Open an empty file named `Dockerfile` in VS Code. Type a comment at the top:

```
# Multi-stage Dockerfile for Python 3.12 Flask app with Gunicorn, non-root user
```

Copilot will suggest the file contents inline. Press `Tab` to accept, or `Alt+]` to cycle through alternatives.

In [ ]:
# Build the Docker image from the generated Dockerfile.
# --progress=plain forces BuildKit to write all output as plain text.
# Without this flag, BuildKit sends TTY-formatted output that is suppressed
# when stdout/stderr are piped (as they are here), so you would see nothing.
# encoding="utf-8" + errors="replace" prevents a UnicodeDecodeError on Windows
# where the default code page (CP1252) cannot decode all BuildKit output bytes.
print("Building image ai-devops-demo:latest ...")
print("This may take 1-3 minutes on first run (downloading the base image).")
print()

build_result = subprocess.run(
    ["docker", "build", "--progress=plain", "-t", "ai-devops-demo:latest", str(OUTPUT_DIR)],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace"
)
print(build_result.stdout or build_result.stderr)

if build_result.returncode != 0:
    print("BUILD FAILED. Error details above.")
    print("Tip: re-run the Dockerfile generation cell, review the output, then retry.")
else:
    print("Build succeeded.")

In [ ]:
# Start the container in detached mode (-d) so it runs in the background.
# Port 5000 on the host maps to port 5000 inside the container.
# --name gives the container a fixed name so we can stop/remove it by name later.
# The cell prints the container ID on success, or the error message on failure.
run_result = subprocess.run(
    ["docker", "run", "-d", "-p", "5000:5000", "--name", "ai-devops-demo", "ai-devops-demo:latest"],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace"
)
output = run_result.stdout.strip() or run_result.stderr.strip()
print(output)

if run_result.returncode != 0:
    print("\nIf the error says 'name already in use', the container from a previous run is still present.")
    print("Run the cleanup cell at the end of this section, then retry.")

In [ ]:
# Smoke test: send an HTTP GET to the running container and verify the response.
# time.sleep(2) gives Gunicorn a moment to start before the first request.
# A successful response looks like: {"service": "ai-devops-demo", "status": "ok"}
# A ConnectionError means the container did not start; check 'docker logs ai-devops-demo'.
import time, requests

time.sleep(2)
try:
    r = requests.get("http://localhost:5000/", timeout=5)
    print(f"HTTP Status: {r.status_code}")
    print(f"Response:   {r.json()}")
    if r.status_code == 200:
        print("\nSmoke test passed. The container is running and serving requests.")
except requests.exceptions.ConnectionError:
    print("Connection refused. The container may still be starting.")
    print("Run this cell again, or check logs with: docker logs ai-devops-demo")

In [ ]:
# Stop and remove the container so the port is free for later cells.
# 'docker rm -f' forces removal even if the container is still running.
result = subprocess.run(
    ["docker", "rm", "-f", "ai-devops-demo"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print("Container removed:", result.stdout.strip() or result.stderr.strip())

## Section 4. Debugging Container Issues with AI

Container build failures are common in real projects: a base image tag is retired, a dependency changes its install path, or a COPY instruction references a file that does not exist in the build context.

This section demonstrates a repeatable debugging workflow:

1. A broken Dockerfile is written to disk intentionally
2. The build is run and the error output is captured into a variable
3. That error output is sent to the AI along with context about what the file is supposed to do
4. The AI returns a root-cause analysis and a corrected line

**What the broken file contains and why it fails**

The base image tag `python:3.99-slim` does not exist on Docker Hub. Python 3.99 has never been released. Docker cannot pull a non-existent image, so the build fails at the very first instruction (`FROM`). This is the same class of error you see when a previously valid tag is deprecated or when a typo is introduced during a version bump.

The error is intentional. The goal of this section is not to avoid the error but to show how to use AI to diagnose and fix it systematically.

In [ ]:
# Write a Dockerfile with a deliberate error: python:3.99-slim does not exist.
# The build will fail. That failure output is what we send to the AI in the next cell.
#
# Expected output from this cell:
#   Docker reports "python:3.99-slim: not found" and exits with a non-zero code.
#   This is the correct, expected behaviour. There is no bug in this cell.
#   The error_log variable now holds the text we need for AI-assisted debugging.
broken_dockerfile = """\
FROM python:3.99-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app.py .
CMD ["gunicorn", "-w", "2", "-b", "0.0.0.0:5000", "app:app"]
"""

broken_path = OUTPUT_DIR / "Dockerfile.broken"
broken_path.write_text(broken_dockerfile)

print("Broken Dockerfile written. Attempting build (this will fail on purpose) ...")
print()

broken_build = subprocess.run(
    ["docker", "build", "-f", str(broken_path), "-t", "ai-devops-broken:test", str(OUTPUT_DIR)],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace"
)

error_log = broken_build.stderr or broken_build.stdout

print("Build exit code:", broken_build.returncode, "(non-zero = failed, which is expected here)")
print()
print("Error output captured (first 1500 characters):")
print("-" * 60)
print(error_log[:1500])
print("-" * 60)
print()
print("This error_log variable is passed to debug_with_ai() in the next cell.")

In [ ]:
# debug_with_ai() is a reusable helper: give it an error log and a description
# of what the file was supposed to do, and it returns:
#   1. Root cause in one sentence
#   2. The corrected line(s)
#   3. How to verify the fix worked
#
# This pattern applies to any Docker error, not just missing base images.
# Copy the helper into your own projects and pass it any build or runtime error log.
def debug_with_ai(error_log: str, context: str) -> str:
    prompt = f"""\
The following Docker build failed.

Context:
{context}

Error log:
{error_log}

Provide:
1. Root cause (one sentence)
2. The corrected line(s)
3. How to verify the fix
"""
    return ask_ai(prompt)

print("Sending error log to Azure OpenAI for analysis ...")
print()

analysis = debug_with_ai(
    error_log=error_log,
    context="Dockerfile for a Python 3.12 Flask app. The base image tag may be incorrect."
)
print(analysis)

**Applying the fix**

The AI should have identified that `python:3.99-slim` does not exist and suggested changing the `FROM` line to `python:3.12-slim`.

To verify: go back to the Dockerfile generation cell in Section 3, confirm it uses `python:3.12-slim`, and re-run the build cell. The build should succeed.

**When to use this pattern in real projects**

- A base image tag is deprecated or renamed (common after major version releases)
- A `RUN` command fails because a package was renamed in the distro's package manager
- A `COPY` fails because the build context does not include the expected file
- A container starts but crashes immediately (pass `docker logs <container>` as the error_log)

## Section 5. Deploying Microservices with AI-Generated Workflows

A real application rarely runs as a single container. This section adds two services alongside the Flask app:

- **Redis** - an in-memory data store the app can use for caching or session storage
- **Nginx** - a reverse proxy that accepts traffic on port 80 and forwards it to the Flask app on port 5000

Docker Compose manages all three services as a unit with a single command. It handles networking between containers automatically: services find each other by name (e.g., the app container connects to `redis:6379`).

We ask the AI to generate the `docker-compose.yml`. The prompt specifies each service's image, port exposure rules, and dependencies so the model has enough context to produce a working file.

In [ ]:
# Generate docker-compose.yml for the three-service stack.
# The prompt uses Compose v2 syntax (no 'version' key at the top) which is
# the current standard. The 'depends_on' fields control startup order:
# Redis starts first, then the app, then Nginx.
compose_prompt = """\
Generate a docker-compose.yml (Compose v2 syntax, no version key) for:

1. Service: app
   - Build from ./Dockerfile
   - Exposes port 5000 internally only (no host port mapping)
   - Depends on: redis

2. Service: redis
   - Image: redis:7-alpine
   - Not exposed externally

3. Service: nginx
   - Image: nginx:alpine
   - Port 80 on the host maps to 80 inside the container
   - Reverse-proxies to app:5000
   - Depends on: app

Include an inline nginx config using a configs block.
Output only the YAML, no explanation.
"""

compose_content = strip_fences(ask_ai(compose_prompt))
(OUTPUT_DIR / "docker-compose.yml").write_text(compose_content)

print("Generated docker-compose.yml:")
print("-" * 60)
print(compose_content)

**Review the Compose file before deploying.**

Check these three things in the generated YAML:

1. **Port exposure:** Only Nginx should have a `ports:` entry (80:80). The app and Redis should have no host port mappings.
2. **Persistence:** If Redis data must survive container restarts, add a named volume (e.g., `redis_data:/data`).
3. **Restart policy:** Add `restart: unless-stopped` to each service for production use.

**Deploy commands** (run these in a terminal on the Linux VM, not in this notebook):

```bash
cd lab_output

# Start all services in the background
docker compose up -d

# Check that all three services are running
docker compose ps

# Tail logs from all services (Ctrl+C to stop)
docker compose logs --tail=50 --follow
```

**Zero-downtime rolling update** (update the app without restarting Redis or Nginx):

```bash
# Rebuild only the app image
docker compose build app

# Replace only the app container; leave Redis and Nginx untouched
docker compose up -d --no-deps app
```

## Section 6. Automating CI/CD with Jenkins Prompt Templates

Prompt templates are Python f-strings that encode your organisation's pipeline conventions. You define them once, fill in per-project variables, and call the AI to produce consistent CI/CD configs across every project.

This section defines three templates and uses the first to generate a complete Jenkinsfile for the Flask app. The second template reviews that Jenkinsfile for common issues. The third is a utility for generating individual pipeline stages.

**How a declarative Jenkinsfile works**

```
pipeline {
    agent any          <- run on any available Jenkins agent
    environment { }   <- inject credentials as environment variables
    stages {
        stage('Build') { steps { ... } }
        stage('Push')  { steps { ... } }
        stage('Deploy') { steps { ... } }
    }
    post {
        always { }    <- runs after every build regardless of outcome
        failure { }   <- runs only when a stage fails
    }
}
```

In [ ]:
# Define three reusable prompt templates as Python f-strings.
# {placeholders} are filled in when the template is used.
#
# JENKINSFILE_TEMPLATE    - generates a complete declarative Jenkinsfile
# PIPELINE_REVIEW_TEMPLATE - asks the AI to critique an existing Jenkinsfile
# STAGE_GENERATOR_TEMPLATE - generates a single named stage block
JENKINSFILE_TEMPLATE = """\
Generate a declarative Jenkinsfile for:
- App: {app_name}
- Registry: {registry}
- Stages: checkout, build Docker image, push to registry
- Use environment variables for credentials (GITHUB_CREDS only)
- The deploy stage should use the Docker registry to push the built image; no SSH deployment step
- Use 'agent any' (not a label like 'windows'); agent must be inside the node block so cleanWs() has file context
- Post block: always run cleanWs() inside the node block; on failure echo a failure message
- Use bat steps (Windows agent), not sh
Output only the Jenkinsfile, no explanation.
"""

PIPELINE_REVIEW_TEMPLATE = """\
Review the following Jenkinsfile and identify:
1. Security issues (hardcoded secrets, missing credential masking)
2. Missing error handling or post-failure steps
3. Performance improvements (parallelism, caching)
Be concise. Format as a numbered list.

Jenkinsfile:
{jenkinsfile_content}
"""

STAGE_GENERATOR_TEMPLATE = """\
Generate a single Jenkins declarative pipeline stage for: {stage_description}
Use shell steps. Output only the stage block.
"""

print("Templates defined: JENKINSFILE_TEMPLATE, PIPELINE_REVIEW_TEMPLATE, STAGE_GENERATOR_TEMPLATE")

In [ ]:
# Jenkinsfile written as a validated template - no AI generation.
# Uses sh steps because Jenkins runs on a Linux container (/var/jenkins_home).
#
# Adjust APP_NAME and REGISTRY before running.

APP_NAME = "ai-devops-demo"
REGISTRY = "myregistry.azurecr.io"

jenkinsfile_content = f"""pipeline {{
    agent any

    environment {{
        IMAGE = "{REGISTRY}/{APP_NAME}:${{BUILD_NUMBER}}"
    }}

    stages {{
        stage('Checkout') {{
            steps {{
                checkout scm
            }}
        }}

        stage('Build Image') {{
            steps {{
                sh "docker build --progress=plain -t $IMAGE ."
            }}
        }}

        stage('Verify Image') {{
            steps {{
                sh "docker images $IMAGE"
            }}  
        }}
    }}

    post {{
        always {{
            cleanWs()
        }}
        failure {{
            echo 'Build failed. Check the Console Output above for details.'
        }}
    }}
}}"""

(OUTPUT_DIR / "Jenkinsfile").write_text(jenkinsfile_content)
print(jenkinsfile_content)


In [ ]:
# Ask the AI to review the Jenkinsfile it just generated.
# This two-step pattern (generate then review) catches common issues
# like missing credential masking or absent post-failure cleanup.
print("Requesting AI review of the generated Jenkinsfile ...")
print()

review = ask_ai(
    PIPELINE_REVIEW_TEMPLATE.format(jenkinsfile_content=jenkinsfile_content)
)
print(review)

**Connecting the Jenkinsfile to Jenkins and verifying the pipeline**

Step 1: Commit the Jenkinsfile to your repository.

```bash
cp lab_output/Jenkinsfile ./Jenkinsfile
git add Jenkinsfile
git commit -m "Add AI-generated Jenkinsfile"
git push
```

Step 2: Create a Pipeline job in Jenkins.

1. Go to the Jenkins dashboard and click **New Item**
2. Enter a name (e.g., `ai-devops-demo`), select **Pipeline**, and click **OK**
3. Scroll down to the **Pipeline** section
4. Set **Definition** to **Pipeline script from SCM**
5. Set **SCM** to **Git**
6. Enter your repository URL under **Repository URL**
7. Set credentials if the repository is private
8. Under **Branches to build**, set the branch to `*/main` (or your default branch)
9. Confirm **Script Path** is set to `Jenkinsfile`
10. Click **Save**


Step 3: Add credentials to Jenkins.

The Jenkinsfile references ro `GITHUB_CREDS`. These must exist in the Jenkins credentials store before the pipeline runs:

1. Go to **Manage Jenkins > Credentials > System > Global credentials > Add Credentials**
2. Add `GTHUB_CREDS` as **Username with password** (registry username + password/token)

Step 4: Verify the pipeline runs correctly.

1. On the pipeline job page, click **Build Now**
2. Open the build and click **Console Output**
3. Confirm each stage completes with `SUCCESS`

```

## Section 7. Terraform Configurations for Production with AI

Terraform is the standard tool for provisioning cloud infrastructure as code. AI is effective for generating boilerplate Terraform configurations, but every file must be reviewed by a human before `terraform apply` is run. AI-generated IaC commonly contains:

- Security group rules open to `0.0.0.0/0` instead of a specific CIDR range
- No remote state backend (state stored locally, which breaks team collaboration)
- Hardcoded values that should be variables
- Missing resource tags required by your organisation's policy

This section generates a `main.tf` and `variables.tf` for an Azure Linux VM configured to host the Docker workload from Section 5.

In [ ]:
# Generate main.tf and variables.tf using a detailed prompt.
# The prompt uses a delimiter convention (# --- main.tf ---) so the response
# can be split into two separate files programmatically.
# strip_fences() removes any HCL code fences the model adds.
tf_prompt = """\
Generate two Terraform files for Azure:

main.tf:
- Provider: azurerm, version ~> 3.0
- Resource group in var.location
- Virtual network and subnet
- Public IP (dynamic)
- Network security group: allow SSH (22) and HTTP (80) inbound only
- Linux VM: Ubuntu 22.04, Standard_B2s, SSH key auth (no password)
- Custom data script that installs Docker CE on first boot
- Resource tags: environment, owner

variables.tf:
- location, resource_group_name, vm_admin_username, ssh_public_key, environment, owner

Output both files clearly delimited with comments like # --- main.tf --- and # --- variables.tf ---
No explanation outside the files.
"""

print("Generating Terraform files ...")
print()

tf_output = ask_ai(tf_prompt)

if "# --- variables.tf ---" in tf_output:
    parts = tf_output.split("# --- variables.tf ---")
    main_tf = strip_fences(parts[0].replace("# --- main.tf ---", ""))
    variables_tf = strip_fences(parts[1])
else:
    main_tf = strip_fences(tf_output)
    variables_tf = "# Delimiter not found - split main.tf manually"

(OUTPUT_DIR / "main.tf").write_text(main_tf)
(OUTPUT_DIR / "variables.tf").write_text(variables_tf)

print("=== main.tf ===")
print(main_tf)
print()
print("=== variables.tf ===")
print(variables_tf)

**Security review checklist - complete before running terraform apply**

1. Are inbound security group rules restricted to a specific CIDR or IP range, not `0.0.0.0/0`?
2. Is the SSH public key stored in a variable, not hardcoded in the resource block?
3. Is there a `backend` block pointing to Azure Blob Storage or Terraform Cloud for remote state?
4. Are all resources tagged with `environment` and `owner`?
5. Does the custom data script use `apt-get install -y docker-ce=<version>` rather than `latest`?


## Section 8. End-to-End Walkthrough: Development to Deployment

Every artifact generated in this lab fits into a single continuous pipeline. This section ties them together and verifies the full set of outputs is present.

**Full pipeline flow**

```
1. Write app code     -->  app.py + requirements.txt  (Section 2)
         |
         v
2. Dockerfile         -->  AI-generated, reviewed, built locally  (Section 3)
         |
         v
3. Local test         -->  docker build + docker run + HTTP smoke test  (Section 3)
         |
         v
4. Compose stack      -->  AI-generated docker-compose.yml  (Section 5)
         |
         v
5. CI/CD              -->  AI-generated Jenkinsfile committed to Git  (Section 6)
         |
         v
6. Infrastructure     -->  AI-generated Terraform, reviewed, applied  (Section 7)
         |
         v
7. Production         -->  Compose stack running on provisioned VM
```

In [ ]:
# List all files generated in this lab with their sizes.
# Every file in this list should have been reviewed before use in a real project.
artifacts = sorted(OUTPUT_DIR.iterdir())
print(f"Generated artifacts in {OUTPUT_DIR}/")
print("-" * 55)
for f in artifacts:
    print(f"  {f.name:35s}  {f.stat().st_size:6d} bytes")
print("-" * 55)
print(f"Total: {len(artifacts)} files")

## Section 9. Best Practices and Wrap-Up

### Prompting tips for DevOps tasks

1. **Give context first** - language, framework, version numbers, cloud provider, and constraints. The more precise the input, the more usable the output.
2. **Constrain the output format** - ending a prompt with "Output only the Dockerfile, no explanation" prevents the model from wrapping the file in prose, which would break tools that parse the file directly.
3. **Ask for explanations separately** - combining generation and explanation in one prompt often degrades both. Generate first, then ask a follow-up question about any part you do not understand.
4. **Iterate in small steps** - generate one stage, review it, then move to the next. Generating an entire pipeline in one shot produces longer output that is harder to verify.
5. **Use system prompts** - the `ask_ai` helper sets a DevOps expert system prompt. Changing this context for specific tasks (e.g., "You are a Terraform security reviewer") shifts the model's focus and improves relevance.

### Security reminders

1. Never paste API keys, passwords, connection strings, or private keys into a prompt. The prompt is logged by the API provider.
2. Always review AI-generated configs before writing them to disk or committing to version control.
3. Treat AI output as a first draft from a junior engineer: useful scaffolding, but not production-ready without review.
4. Use a `.env` file locally and a secrets manager (Azure Key Vault, AWS Secrets Manager, HashiCorp Vault) in CI/CD. Never hardcode credentials in Dockerfiles, Jenkinsfiles, or Terraform.

### When NOT to trust AI output

- Terraform state file management and backend configuration
- IAM policies and role bindings (a single misconfiguration can expose your entire environment)
- Network security group rules for production (verify every port rule manually)
- Any configuration that touches production data stores or backups
- Compliance-sensitive settings: encryption at rest, audit logging, retention policies, GDPR/HIPAA controls

In these areas, use AI to understand your options and generate a starting point, then apply manual expert review and your organisation's security baseline before applying any changes.